In [ ]:
import os
import joblib
from functions import *
import matplotlib.pyplot as plt

In [ ]:
# Load Dataset for evaluation

n_trace = 10000
dt_ = 25e-9  #sampling rate in seconds. 40MHz
# base_noise = 0  # add in during testing for flexibility...

vmin = 1
vmax = 3
v_n = 100
rmin = 6
rmax = 9
r_n = 100
baseline = 100 # mV
trace_type = 'const'
pulse_type = 'nai'

base = ('data_{}_{}_{}_{}_{}_{}_{}_{}_{}_{}_{}_'
        .format(n_trace, dt_, trace_type, pulse_type, baseline,
                vmin, vmax, v_n, rmin, rmax, r_n))
base = os.path.join(os.getcwd(), 'data', base)
print('Loading Data. This may take a while')
X = joblib.load(base + 'X.pkl')
Y = joblib.load(base + 'Y.pkl')
V = joblib.load(base + 'V.pkl')
R = joblib.load(base + 'R.pkl')
photons = joblib.load(base + 'photons.pkl')
print('Data Loaded')

assert V.shape == R.shape == X.shape[0:2] == Y.shape[0:2] == (len(photons), len(photons[0]))

# Set parameters for NAI FPGA trace to counts function
#variables for integrating trace pulses into listmode events
thresh = 8.0     #units of mV  this is the pulse trigger threshold
int_i = 50      #integ.ration time = 1.25 microsecs = 50 samples at 40MHz sampling 
dead_i = int_i     #deadtime = integration time
extend = 1    #extendable dead time parameter
escale = .63  #being used to scale the pulse integration value to energy in keV. experimentally determined.


In [ ]:
def percent_photons_counted(truth_times_, count_times_):
	return count_times_.size / truth_times_.size

def percent_coincident_photons(truth_times_):
	# these percent are uncountable as they are summed before any other processing
	_, counts = np.unique(truth_times_, return_counts=True)
	# set all indeces occuring once to zero, probably most for low to med count rates
	counts -= 1
	# counts will be >= 0
	return np.sum(counts) / truth_times_.size

def percent_saturation(x_):
	assert len(x_.shape) == 1
	return np.where(x_ >= 1000)[0].size / x_.size

def percent_energy_counted_listmode(truth_energies, est_energies):
	return np.sum(est_energies) / np.sum(truth_energies)

def error_energy_listmode(truth_energies, est_energies):
	return np.sum(truth_energies) - np.sum(est_energies)
	
def error_energy_timeseries(truth_series, est_series):
	return np.sum(truth_series, est_series)

def prob_photon_per_index(truth_times_, x_):
	#BAYES? TODO
	
	# true energies are sampled uniformly with replacement
	# from sample bins
	
	p_index_i_per_photon = 1 / x_.size
	photon = 1 # because we generated it?
	
	assert len(x_.shape) == 1
	return x_.size

In [ ]:
i = 1
j = 1

x = X[i, j]
y = Y[i, j]
v = V[i, j]
r = R[i, j]
truth_times = photons[i][j] # All magnitude v with constant trace

# Note that dt in this f is seconds before pulse
# tstep is "conventional dt"
count_energies, count_times = trace_to_counts(x, dt=0, tstep=dt_, thresh=thresh, baseline=baseline,
								  extend=extend, escale=escale, int_i=int_i, dead_i=dead_i)

print(y)

print('OLD FPGA Algo')
pct_p_counted = percent_photons_counted(truth_times, count_times)
print('pct_counted', pct_p_counted)
pct_true_coincident = percent_coincident_photons(truth_times)
print('pct_true_coincident', pct_true_coincident)
pct_sat = percent_saturation(x)
print('pct_sat', pct_sat)

# these two are also a little funky, its a little apples and oranges
# energy from trace_to_counts is integrated voltage and multiplied by const
# 
pct_energy_counted = percent_energy_counted_listmode(truth_times, count_energies) # TODO this might need investigation
print('pct_energy_counted', pct_energy_counted)
error_energy_counted = error_energy_listmode(truth_times, count_energies) # TODO this might need investigation
# print('error_energy_counted', error_energy_counted)

fig = plt.figure(figsize=(10, 8), dpi=400)
plot_photons(fig, count_times, 1, 0, 'r', label_='Counted')
plot_photons(fig, truth_times, 1, 1, 'b', label_='Truth')
plt.legend()
plt.title('{:.3f}% Counted'.format(pct_p_counted*100))
plt.show()
